In [ ]:
# ============================================================
# Notebook 02: "The Right Angle That Explains Everything"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    # Running in Colab or fresh environment — install from GitHub
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/YOUR_USERNAME/regression-geometry-course.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import matplotlib.pyplot as plt

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence

from regression_geometry.core import ColumnSpace, Projection, HatMatrix
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard

# --- Reproducibility ---
np.random.seed(42)

> *"The right angle is the most important angle in statistics."*

## Where We Are

In Notebook 1, you saw that OLS drops a perpendicular from y to the column space. The residual hits the plane at a right angle.

This notebook asks: so what? What does that right angle actually guarantee?

The answer is: almost everything.

In [ ]:
# Recap: the n=3 projection from Notebook 1
x3 = np.array([2.0, 5.0, 8.0])
y3 = np.array([6.5, 12.8, 19.2])
cs3 = ColumnSpace(x3, add_intercept=True)
proj3 = cs3.project(y3)
e3 = proj3.residuals
X3 = cs3.X

# Verify X'e = 0
print(f"Column 1 (intercept) \u00b7 e = {X3[:, 0] @ e3:.10f}")
print(f"Column 2 (x)         \u00b7 e = {X3[:, 1] @ e3:.10f}")
print()
print("When textbooks write X'e = 0 (the 'normal equations'), they're saying:")
print("the residual is perpendicular to every column of X. That's the whole content.")

---
### 🛑 PREDICT FIRST

If you project y onto the plane to get ŷ, and then project ŷ again — what do you get? Something different? Something closer? The same thing?

**Before running the next cell, write your prediction below:**

What is Hŷ equal to, where H is the hat matrix?

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# Project y to get y_hat, then project y_hat again
H3 = cs3.hat_matrix()
y_hat = H3 @ y3
y_hat_again = H3 @ y_hat  # project the shadow

print(f"y_hat           = {y_hat}")
print(f"H @ y_hat       = {y_hat_again}")
print(f"Difference:       {np.max(np.abs(y_hat - y_hat_again)):.2e}")
print()
print("Identical. The shadow of a shadow is itself.")
print("In matrix language: H\u00b2 = H. The hat matrix is 'idempotent.'")

hm3 = HatMatrix(H3)
print(f"\nVerify H\u00b2 = H: {hm3.verify_idempotent()}")

This isn't a weird algebraic trick. It's obvious from the picture: if ŷ is already ON the plane, dropping a perpendicular from it just lands on itself.

In [ ]:
# The hat matrix is also symmetric: H = H'
print(f"Verify H = H': {hm3.verify_symmetric()}")
print()
print("The hat matrix is symmetric: H = H'.")
print("The projection treats all directions equally —")
print("it doesn't distort the space, it just flattens onto the plane.")

## The Hat Matrix as a Machine

The matrix H = X(X'X)⁻¹X' is called the "hat matrix" because it puts the hat on y — it turns y into ŷ. But it's more than a formula. It's a geometric machine: feed it any vector, and it returns the shadow of that vector on the column space.

In [ ]:
# Feed H several random vectors — each output lands on the column space
rng = np.random.default_rng(42)
print("Input vector         →  H @ input (shadow)        On the plane?")
print("-" * 70)
for i in range(4):
    v = rng.normal(size=3) * 10
    shadow = H3 @ v
    coeffs = np.linalg.lstsq(X3, shadow, rcond=None)[0]
    reconstruction = X3 @ coeffs
    on_plane = np.allclose(shadow, reconstruction)
    print(f"  {v.round(2)}  →  {shadow.round(2)}  {on_plane}")

## What the Right Angle Guarantees

Here's why the right angle matters for practice. X'e = 0 means: the residuals have zero correlation with every predictor. Not approximately zero. EXACTLY zero.

This is true BY CONSTRUCTION — it's a property of the projection, not evidence that your model is correct.

In [ ]:
# On the Meridian dataset: residuals are exactly uncorrelated with predictors
df = load_meridian()
X_mer = sm.add_constant(df["experience"])
model_mer = sm.OLS(df["salary"], X_mer).fit()

corr_resid_exp = np.corrcoef(model_mer.resid, df["experience"])[0, 1]
print(f"Correlation between residuals and experience: {corr_resid_exp:.10f}")
print(f"X'e = {(X_mer.T @ model_mer.resid).values.round(8)}")
print()
print("Exactly zero. By construction.")

---
### 🛑 PREDICT FIRST

Does the fact that residuals are uncorrelated with predictors mean the model is correct?

**Before running the next cell, write your prediction below:**

What could still be wrong even though X'e = 0?

---

In [ ]:
# YOUR PREDICTION:
#
#

## The Dangerous Misinterpretation

No. Residuals are perpendicular to the column space by CONSTRUCTION. This tells you NOTHING about whether your model is correct.

You could regress salary on zodiac sign and the residuals would still be perpendicular to the zodiac column. The right angle is a property of the method, not a validation of the model.

This is one of the most common misunderstandings in applied regression.

In [ ]:
# Prove it: regress salary on pure noise
rng = np.random.default_rng(99)
noise = rng.normal(size=len(df))
X_noise = sm.add_constant(noise)
model_noise = sm.OLS(df["salary"], X_noise).fit()

print(f"Regression of salary on random noise:")
print(f"  R\u00b2 = {model_noise.rsquared:.6f}  (basically zero)")
print(f"  X'e = {(X_noise.T @ model_noise.resid).round(8)}")
print()
print("The right angle holds even though the model is nonsense.")

## What Residuals CAN Tell You

Residuals can't validate your model, but they can reveal patterns the model MISSED. If you plot residuals against a variable NOT in the model and see a pattern, that's a signal.

In [ ]:
# Fit salary ~ experience (omitting education), plot residuals vs education
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: residuals vs experience (in model) — no pattern
ax1.scatter(df["experience"], model_mer.resid, alpha=0.3, s=15, color=RESIDUAL)
ax1.axhline(0, color=SECONDARY, ls="--")
ax1.set(xlabel="Experience (in model)", ylabel="Residuals",
        title="Residuals vs predictor IN model")

# Right: residuals vs education (NOT in model) — visible pattern
ax2.scatter(df["education"], model_mer.resid, alpha=0.3, s=15, color=RESIDUAL)
ax2.axhline(0, color=SECONDARY, ls="--")
ax2.set(xlabel="Education (NOT in model)", ylabel="Residuals",
        title="Residuals vs variable NOT in model")

plt.tight_layout()
plt.show()

In [ ]:
# Interactive: see the projection and orthogonality in the n=3 example
fig = viz.plot_projection_3d(cs3, y3, title="Verify: the right angle holds")

## The Residual Maker Matrix

If H puts the hat on y (producing ŷ), then I − H takes the hat off — it produces the residuals: e = (I − H)y.

This matrix M = I − H is called the "residual maker" or "annihilator" — it annihilates the component of y that lies on the plane and keeps only the perpendicular part.

In [ ]:
# Verify M = I - H is also idempotent and symmetric
n = len(y3)
M3 = np.eye(n) - H3
hm_M3 = HatMatrix(M3)

print(f"M\u00b2 = M?  {hm_M3.verify_idempotent()}")
print(f"M = M'?  {hm_M3.verify_symmetric()}")
print(f"My = e?  {np.allclose(M3 @ y3, e3)}")
print()
print("M is also a projection — onto the space PERPENDICULAR to the column space.")
print("H and M together decompose any vector into two perpendicular parts.")

<details>
<summary><b>Going Deeper:</b> Why M is also a projection</summary>

M = I − H is the projection onto the orthogonal COMPLEMENT of the column space. Together, H and M decompose any vector into two perpendicular components: the part on the plane (Hy = ŷ) and the part perpendicular to it (My = e).

Since Hy and My are perpendicular and they sum to y, the squared lengths must satisfy the Pythagorean theorem: ‖y‖² = ‖ŷ‖² + ‖e‖². This is exactly SST = SSR + SSE — which is Notebook 3's topic.

</details>

---
### 🛑 PREDICT FIRST

You're about to run a regression with four predictors on 2,000 observations. The hat matrix will be 2000×2000.

**Before running the next cell, write your prediction below:**

Will X'e still be exactly zero with four predictors? Or does the orthogonality break down with more predictors?

---

In [ ]:
# YOUR PREDICTION:
#
#

---
### 🛑 DIAGNOSE FIRST

Here's a multiple regression on Meridian: salary ~ experience + education + performance.

**Before seeing the visualization, answer:**

The hat matrix for this model is 2000×2000. What is its trace? (Hint: think about what tr(H) counts.)

---

In [ ]:
# YOUR ANSWER:
# tr(H) = ?
#

In [ ]:
# Run salary ~ experience + education + performance + gender
X_full = sm.add_constant(df[["experience", "education", "performance", "gender"]])
model_full = sm.OLS(df["salary"], X_full).fit()
print(model_full.summary().tables[1])

# Verify orthogonality
ortho_check = X_full.T @ model_full.resid
print(f"\nX'e (should all be ~0):")
print(f"  {ortho_check.values.round(6)}")

# Trace of H
leverage = OLSInfluence(model_full).hat_matrix_diag
print(f"\ntr(H) = {leverage.sum():.4f} (= number of columns in X: {X_full.shape[1]})")

Four predictors plus intercept means the column space is a 5-dimensional "plane" in ℝ²⁰⁰⁰. You can't draw it. But the right angle is there — the residual is perpendicular to all five directions. Every property demonstrated in 3D works identically here.

---
### 🔗 Back to Statsmodels

| Geometric concept | Code |
|---|---|
| Hat matrix diagonal hᵢᵢ | `OLSInfluence(model).hat_matrix_diag` |
| Residual maker (I−H)y | `model.resid` |
| Idempotency check H² = H | `HatMatrix(H).verify_idempotent()` |
| Trace of H | `OLSInfluence(model).hat_matrix_diag.sum()` |

---

In [ ]:
# Demonstrate on the Meridian multiple regression
print("Hat matrix diagonal (first 10):")
print(f"  {leverage[:10].round(4)}")
print(f"\nAverage leverage: {leverage.mean():.4f}  (= k/n = {X_full.shape[1]}/{len(df)})")
print(f"\nResidual (first 5): {model_full.resid.values[:5].round(0)}")
print(f"\nVerify e \u22a5 X:")
print(f"  X'e = {(X_full.T @ model_full.resid).values.round(6)}")

---
### ✍️ The Memo

You're writing a memo to Meridian's Head of Data Science.

In 3 sentences, explain why the fact that residuals are uncorrelated with predictors does NOT mean the model is correctly specified. Give a concrete example of how the model could be wrong despite this property.

**Rules:** Do not use the words "orthogonal," "hat matrix," "idempotent," or "projection." Write in plain English.

---

In [ ]:
# YOUR MEMO:
#
#
#

In [ ]:
# Geometric Scoreboard — unlock tr(H)/n gauge
cs_full = ColumnSpace(X_full.values, add_intercept=False)
proj_full = cs_full.project(df["salary"].values)

scoreboard = GeometricScoreboard(
    proj=proj_full,
    cs=cs_full,
    active_gauges=["theta", "r_squared", "leverage"],
    mode="widget" if INTERACTIVE else "static"
)
scoreboard.display()

A new gauge is unlocked: tr(H)/n, the average leverage. It always equals k/n — the number of predictors divided by the sample size. It tells you how much "room" the model has to fit the data.

---
### Summary

**What you learned:**
- The hat matrix H is a projection operator: it's idempotent (H² = H) and symmetric (H = H').
- Residuals are perpendicular to predictors BY CONSTRUCTION — this is not model validation.
- The annihilator M = I − H extracts the perpendicular component of any vector.

**Key geometric insight:** ***X'e = 0 is the definition of the projection, not evidence that the model is correct.***

### Next

That right angle creates a right TRIANGLE — and right triangles obey the Pythagorean theorem. Notebook 3 shows that the most famous equation in statistics, SST = SSR + SSE, is literally Pythagoras.

---